# SteerMoE expert detection — vector construction

Replicates the detection phase of **"Steering MoE LLMs via Expert (De)Activation"**
([arXiv:2509.09660](https://arxiv.org/abs/2509.09660),
[official code](https://github.com/adobe-research/SteerMoE)) — the official
`custom_steering.ipynb` demo — with EasySteer's `router_logits` capture
stream, no forked model code needed:

1. Capture per-token router logits for a few **contrastive pairs**
   (answering with digits `1, 2, 3` vs. words `one, two, three`).
2. Compute each expert's top-k selection rate on the behavior tokens and
   the **risk difference** `Δ = p_digits − p_words`.
3. Save the top word-linked experts as a `deactivate` steering config for
   `steermoe_steer.ipynb`.

Model: `Qwen/Qwen3-30B-A3B` (48 MoE layers × 128 experts, top-8),
one of the models evaluated in the paper.

In [1]:
import json
import os

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
# Qwen3-MoE is outside the default V2-runner list; capture
# requires the V2 runner (exact batch geometry for row labels).
os.environ.setdefault("VLLM_USE_V2_MODEL_RUNNER", "1")

import numpy as np
from vllm import LLM, SamplingParams
from vllm.capture import deserialize_captured

MODEL = "/data/zju-130/shenyl/hf/model/Qwen/Qwen3-30B-A3B"  # Qwen/Qwen3-30B-A3B

with open(os.path.join(MODEL, "config.json")) as f:
    hf_cfg = json.load(f)
N_EXPERTS = hf_cfg["num_experts"]      # 128
TOP_K = hf_cfg["num_experts_per_tok"]  # 8

# Router-logit capture uses gate forward hooks, so the engine must run
# eagerly; prefix caching stays off because cache-hit tokens are never
# recomputed and so could never be captured.
llm = LLM(
    model=MODEL,
    enforce_eager=True,
    tensor_parallel_size=1,
    enable_chunked_prefill=False,
    enable_prefix_caching=False,
    gpu_memory_utilization=0.92,
    max_model_len=4096,
)
tok = llm.get_tokenizer()


def rpc(method, *args, **kwargs):
    return llm.llm_engine.collective_rpc(method, args=args, kwargs=kwargs)[0]

/home/xhl/anaconda3/envs/easysteer-vllm026/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/xhl/anaconda3/envs/easysteer-vllm026/lib/python3.12/site-packages/pydantic/dataclasses.py:313: UserWarning: `config` is set via both the `dataclass` decorator and `__pydantic_config__` for dataclass SteerVectorConfig. The `config` specification from `dataclass` decorator will take priority.
  return create_dataclass if _cls is None else create_dataclass(_cls)


INFO 08-04 01:37:50 [api_utils.py:273] non-default args: {'max_model_len': 4096, 'enable_prefix_caching': False, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'model': '/data/zju-130/shenyl/hf/model/Qwen/Qwen3-30B-A3B'}


INFO 08-04 01:37:50 [model.py:623] Resolved architecture: Qwen3MoeForCausalLM


INFO 08-04 01:37:50 [model.py:1788] Using max model len 4096


...

WARNING 08-04 01:48:45 [vllm.py:1249] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.


(EngineCore pid=4068530) 

INFO 08-04 01:48:45 [kernel.py:295] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])


(EngineCore pid=4068530) 

INFO 08-04 01:48:45 [vllm.py:1428] Cudagraph is disabled under eager mode


## 1. The contrastive pairs

Each side renders a full chat turn **including the assistant response**, so
a single prefill routes every response token through the MoE layers. The
`target` string marks the tokens whose routings we compare. The official
demo uses a single pair; a few pairs sharpen the risk difference
considerably.

In [2]:
PAIRS = [
    ("Count to ten",
     "1, 2, 3, 4, 5, 6, 7, 8, 9, 10",
     "one, two, three, four, five, six, seven, eight, nine, ten"),
    ("How many days are in a week, and how many months in a year?",
     "There are 7 days in a week and 12 months in a year.",
     "There are seven days in a week and twelve months in a year."),
    ("What is five plus three?",
     "5 + 3 = 8",
     "five plus three equals eight"),
]

## 2. Capture router logits

`start_capture("router_logits")` just turns the stream on — the capture
hooks already sit on every MoE gate. One prefill later, `fetch_captured`
returns `{layer: (num_tokens, n_experts)}`.

In [3]:
def find_sub_list(sub, seq):
    n = len(sub)
    return [(i, i + n - 1) for i in range(len(seq) - n + 1)
            if seq[i:i + n] == sub]


def topk_membership(rows):
    """(tokens, n_experts) logits -> bool top-k membership mask."""
    order = np.argsort(rows, axis=-1)[:, -TOP_K:]
    mask = np.zeros(rows.shape, dtype=bool)
    np.put_along_axis(mask, order, True, axis=-1)
    return mask


counts = {"digits": None, "words": None}
totals = {"digits": 0, "words": 0}
for user, digits_ans, words_ans in PAIRS:
    for key, answer in (("digits", digits_ans), ("words", words_ans)):
        msgs = [{"role": "user", "content": user},
                {"role": "assistant", "content": answer}]
        text = tok.apply_chat_template(msgs, tokenize=False,
                                       add_generation_prompt=False,
                                       enable_thinking=False)
        prompt_ids = tok(text, add_special_tokens=False).input_ids

        rpc("start_capture", "router_logits")
        llm.generate({"prompt_token_ids": prompt_ids},
                     sampling_params=SamplingParams(temperature=0.0,
                                                    max_tokens=1))
        logits = {lid: t.float().numpy()
                  for lid, t in deserialize_captured(
                      rpc("fetch_captured", "router_logits"))[0].items()}
        rpc("stop_capture", "router_logits")

        # top-k selection counts on the target tokens only
        target_ids = tok(answer, add_special_tokens=False).input_ids
        s, e = find_sub_list(target_ids, prompt_ids)[-1]
        sel = np.stack([topk_membership(logits[lid][s:e + 1])
                        for lid in sorted(logits)])
        cnt = sel.sum(axis=1)  # (layer, expert)
        counts[key] = cnt if counts[key] is None else counts[key] + cnt
        totals[key] += e - s + 1

print(f"detection tokens: digits={totals['digits']} "
      f"words={totals['words']}")

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Rendering prompts: 100%|██████████| 1/1 [00:00<00:00, 46.36it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  5.66it/s, est. speed input: 260.24 toks/s, output: 5.66 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  5.66it/s, est. speed input: 260.24 toks/s, output: 5.66 toks/s]

...

Rendering prompts: 100%|██████████| 1/1 [00:00<00:00, 987.36it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00, 14.88it/s, est. speed input: 372.35 toks/s, output: 14.89 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00, 14.59it/s, est. speed input: 372.35 toks/s, output: 14.89 toks/s]

detection tokens: digits=53 words=38


## 3. Risk difference

`Δ(layer, expert) = p_digits − p_words`: experts with large positive Δ are
selected for digit tokens but not word tokens.

In [4]:
risk_diff = counts["digits"] / totals["digits"] \
    - counts["words"] / totals["words"]

flat = np.argsort(np.abs(risk_diff), axis=None)[::-1]
print("top behavior-linked experts (layer, expert, Δ):")
for idx in flat[:10]:
    layer, expert = divmod(int(idx), N_EXPERTS)
    print(f"  L{layer:02d} E{expert:02d}  "
          f"Δ={risk_diff[layer, expert]:+.2f}")

top behavior-linked experts (layer, expert, Δ):
  L42 E30  Δ=-0.51
  L45 E20  Δ=-0.50
  L05 E22  Δ=-0.50
  L40 E115  Δ=-0.46
  L00 E85  Δ=+0.45
  L43 E75  Δ=-0.44
  L03 E54  Δ=-0.44
  L00 E34  Δ=-0.42
  L04 E32  Δ=-0.42
  L17 E49  Δ=-0.41


## 4. Save the steering config

Deactivating the **word-linked** experts (negative Δ) steers *away from
spelled-out numbers*: greedy decoding switches from "One, two, three…"
to "1, 2, 3…". On Qwen3-30B-A3B this direction is strong and saturates —
200 deactivated experts (~3% of 128×48) flips every test prompt to pure
digits, and the effect is stable across reruns. (The digit-linked
direction is much weaker on this model. Fewer experts only perturb
phrasing; many more degrade generation — the paper tunes this count per
model and task, Table A.2.)


In [5]:
N_DEACT = 200

deact = {}
taken = 0
for idx in flat:
    layer, expert = divmod(int(idx), N_EXPERTS)
    # word-linked experts have negative delta (digits_rate - words_rate)
    if risk_diff[layer, expert] >= 0:
        continue
    deact.setdefault(layer, []).append(expert)
    taken += 1
    if taken == N_DEACT:
        break

with open("steermoe_qwen3_words.json", "w") as f:
    json.dump({"layer_configs": {
        str(layer): {"mode": "deactivate", "expert_ids": ids}
        for layer, ids in deact.items()
    }}, f, indent=2)
print(f"saved steermoe_qwen3_words.json: {taken} experts "
      f"across {len(deact)} layers")

saved steermoe_qwen3_words.json: 200 experts across 46 layers
